# Life-cycle OLG with borrowing constraints — the two-stage-loss model

The fourth `examples/` stop, and the one that motivated a new capability in the
framework. This is **Day 2, Exercise 4** of the Geneva 2026 *Deep Learning for
Economics & Finance* course: a 6-generation overlapping-generations economy
where households live $H = 6$ deterministic periods (one period $\approx$ 10
years, ages 20–80), save in capital subject to a **borrowing constraint**
$k^h_t \ge 0$, and supply age-dependent labor (less in the last two periods —
retirement).

Each working cohort $h \in \{0, \dots, H-2\}$ chooses a saving rate; the last
cohort consumes everything. The optimality condition is an intertemporal Euler
that holds with equality **only when the borrowing constraint is slack**:

$$\frac{1}{c^h_t} \;\ge\; \beta\,\mathbb{E}_t\!\left[\frac{1-\delta+r_{t+1}}{c^{h+1}_{t+1}}\right],
\qquad k^{h+1}_{t+1}\ge 0,\qquad \text{complementary slackness.}$$

We encode the complementarity with a Fischer-Burmeister residual, exactly as the
course notebook does.

## Why this model needed a new loss

The FB nonlinearity wraps an **expectation**. And $\mathbb{E}[f^{FB}(\cdot)] \ne
f^{FB}(\mathbb{E}[\cdot])$: averaging the residual over shocks and *then* squaring
(the standard DEQN path) puts the nonlinearity in the wrong place and leaves a
bias floor. So this model is trained with the framework's **two-stage loss**:

- `inside_fn` returns the shock-dependent continuation terms
  $(1-\delta+r')/c'^{\,j}$, which the loss averages to $\mathbb{E}[\cdot]$;
- `combine_fn` applies the Fischer-Burmeister **after** the expectation.

The standard $(\mathbb{E}[r])^2$ path is the special case `combine = identity`.
This is the architectural prize from the port: occasionally-binding constraints
**under uncertainty**, solved MC-correctly.

**Outline**
- 1 — Inspect the model
- 2 — Train (two-stage loss, no closed-form steady state)
- 3 — Loss curve
- 4 — Reproduce the Exercise-4 diagnostic panels
- 5 — Ergodic accuracy + the borrowing-constraint corner
- 6 — Summary

In [ ]:
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np

from deqn_jax.config import TrainConfig
from deqn_jax.models.olg_lifecycle import MODEL
from deqn_jax.models.olg_lifecycle.equations import _cohort_block, combine_fn, inside_fn
from deqn_jax.models.olg_lifecycle.variables import CONSTANTS, H, L_CYCLE
from deqn_jax.plots import plot_loss_curve
from deqn_jax.training.trainer import train_from_config

## 1. Inspect the model

State $\mathbf{x}_t = (Z_t,\, k^0_t, \dots, k^5_t)$ — TFP plus the capital held by
each age group (7 states). Policy is the saving rate out of cash-at-hand for the
five cohorts that still save (sigmoid-bounded to $(0,1)$, so consumption and
saved capital are both positive by construction). Five Euler conditions, one per
saving cohort.

There is **no closed-form steady state**; the cross-sectional capital
distribution and TFP are solved jointly over the ergodic distribution, seeded
from a random $\exp(\mathcal{U}(0,1))$ init — just like the course notebook.

In [ ]:
print(f"states   : {MODEL.n_states}  {MODEL.state_names}")
print(f"policies : {MODEL.n_policies}  {MODEL.policy_names}")
print(f"equations: {len(MODEL.equation_names)}  {MODEL.equation_names}")
print(f"two-stage hooks present: inside_fn={MODEL.inside_fn is not None}, "
      f"combine_fn={MODEL.combine_fn is not None}")
print(f"steady_state_fn: {MODEL.steady_state_fn}  (None -> trained from random init)")
print()
print(f"alpha={CONSTANTS['alpha']}, beta={CONSTANTS['beta']:.4f} (=0.99^10), "
      f"delta={CONSTANTS['delta']}, rho_z={CONSTANTS['rho_z']:.4f}, sigma_z={CONSTANTS['sigma_z']}")
print(f"age-dependent labor l_cycle = {L_CYCLE}  (retirement in the last two periods)")

## 2. Train

Recipe from `configs/olg_lifecycle.yaml`: a `[70, 70]` ReLU MLP with sigmoid
output (Simon's `10 * n_input` width), Adam at `3e-4`, MC expectation with
antithetic shocks. The two-stage path is selected automatically because the
model declares `inside_fn` + `combine_fn`. A few thousand episodes converge in
seconds on CPU.

In [ ]:
cfg = TrainConfig.from_yaml("../configs/olg_lifecycle.yaml")
params, history = train_from_config(cfg)

## 3. Loss curve

The loss is the mean squared Fischer-Burmeister residual across the five Euler
conditions, with the expectation taken *inside* the FB.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
plot_loss_curve(history, ax=ax, log_y=True)
ax.set_title("olg_lifecycle — two-stage FB-Euler loss (log scale)")
plt.show()
print(f"loss: {float(history['loss'][0]):.3e} -> {float(min(history['loss'][-50:])):.3e}")

## 4. Reproduce the Exercise-4 diagnostic panels

The course notebook reads a trained policy off by plotting, **by age group**, the
cross-sectional capital, consumption, and cash-at-hand profiles; the return
distribution; the **signed relative Euler error** `errREE`; the **Euler LHS vs
RHS** check; and the saving policy. We reproduce that exact panel set on the
deqn-jax solution, using Simon's own quantities.

First, simulate the ergodic distribution under the trained policy, then compute
the per-cohort quantities with the model's own `_cohort_block`. The relative
Euler error is the MC-correct two-stage residual — average the continuation
terms over shocks, then apply Fischer-Burmeister (reusing `inside_fn` +
`combine_fn`):

$$\text{err}_{REE}^h = f^{FB}\!\Big(\underbrace{\tfrac{1}{c^h\,\beta\,\mathbb{E}[(1-\delta+r')/c'^{\,h+1}]} - 1}_{a:\ \text{Euler gap}},\ \underbrace{\tfrac{\text{sav}^h}{c^h}}_{b:\ \text{normalized slack}}\Big),$$

with the Euler **LHS** $= 1/c^h$ and **RHS** $= \beta\,\mathbb{E}[(1-\delta+r')/c'^{\,h+1}]$.
This is the exact `errREE` / LHS / RHS triple from the Day 2 Exercise 4 cost
function.

> **Reading the LHS/RHS panel.** A visible **LHS $>$ RHS** gap at the youngest
> cohort ($h=0$) is *expected, not a solver error*: the borrowing constraint
> binds there ($k'\approx 0$), so the Euler inequality holds with slack rather
> than equality — faithful to Simon's reference. LHS tracks RHS for the older,
> unconstrained cohorts.

In [ ]:
def simulate(params, key, n_traj=1024, T=80, burn=30):
    """Roll the trained policy forward; collect post-burn-in ergodic states."""
    s = MODEL.init_state_fn(key, n_traj, CONSTANTS)
    visited = []
    for t in range(T):
        key, sk = jr.split(key)
        eps = jr.normal(sk, (n_traj, 1))
        if t >= burn:
            visited.append(s)
        s = MODEL.step_fn(s, params(s), eps, CONSTANTS)
    return jnp.concatenate(visited, axis=0)


def ergodic_diagnostics(params, X, key, n_shocks=64):
    """MC-correct Exercise-4 diagnostics: average the H continuation terms over
    shocks, then form the SIGNED relative Euler error errREE = fb(a, b) plus the
    Euler LHS / RHS, exactly as the course cost function does.

        a = 1/(c^h * beta * E[(1-delta+r')/c'^{h+1}]) - 1   (Euler gap)
        b = sav^h / c^h                                      (normalized slack)
        LHS = 1/c^h            RHS = beta * E[(1-delta+r')/c'^{h+1}]
    """
    insides = []
    for _ in range(n_shocks):
        key, sk = jr.split(key)
        eps = jr.normal(sk, (X.shape[0], 1))
        ns = MODEL.step_fn(X, params(X), eps, CONSTANTS)
        ins = inside_fn(X, params(X), ns, params(ns), CONSTANTS)
        insides.append(jnp.stack([ins[f"inside_{j}"] for j in range(H)], axis=1))
    E = jnp.mean(jnp.stack(insides), axis=0)                      # [n, H]
    Edict = {f"inside_{j}": E[:, j] for j in range(H)}
    res = combine_fn(X, params(X), Edict, CONSTANTS)
    errREE = jnp.stack([res[f"euler_{h}"] for h in range(H - 1)], axis=1)  # signed
    blk = _cohort_block(X[:, :1], X[:, 1 : 1 + H], params(X), CONSTANTS)
    LHS = 1.0 / blk["c"][:, : H - 1]                  # 1/c^h
    RHS = CONSTANTS["beta"] * E[:, 1:H]               # beta E[(1-d+r')/c'^{h+1}]
    return np.asarray(errREE), np.asarray(LHS), np.asarray(RHS)


X = simulate(params, jr.PRNGKey(1))
Z, k = X[:, :1], X[:, 1 : 1 + H]
blk = _cohort_block(Z, k, params(X), CONSTANTS)
c, sav, cah, r = (np.asarray(blk[n]) for n in ("c", "sav", "cah", "r"))
k_np = np.asarray(k)
errREE, LHS, RHS = ergodic_diagnostics(params, X, jr.PRNGKey(2))
print(f"ergodic sample: {X.shape[0]} states")

In [ ]:
def band(ax, data, title, ylabel, xl="age group"):
    """mean/min/max life-cycle profile on its natural (level) scale."""
    a = np.arange(data.shape[1])
    ax.plot(a, data.mean(0), "o-", label="mean")
    ax.plot(a, data.min(0), "--", alpha=0.4, label="min")
    ax.plot(a, data.max(0), "--", alpha=0.4, label="max")
    ax.set_title(title); ax.set_xlabel(xl); ax.set_ylabel(ylabel); ax.legend(fontsize=8)


fig, ax = plt.subplots(2, 4, figsize=(18, 8))

# life-cycle VALUE profiles -- levels, shown on their natural scale
band(ax[0, 0], k_np, "capital by age group", "capital held")
band(ax[0, 1], c, "consumption by age group", "consumption")
band(ax[0, 2], cah, "cash-at-hand by age group", "cash-at-hand")

# return distribution
ax[0, 3].hist(r[:, 0], bins=40, color="steelblue")
ax[0, 3].set_title("return distribution")
ax[0, 3].set_xlabel("return r_t"); ax[0, 3].set_ylabel("count")

# SIGNED relative Euler error: an error centered on zero -> keep it near zero
# with a zero reference line (do NOT autoscale a log|.| into noise).
a = np.arange(errREE.shape[1])
ax[1, 0].axhline(0.0, color="grey", lw=0.8, zorder=0)
ax[1, 0].plot(a, errREE.mean(0), "o-", label="mean")
ax[1, 0].plot(a, errREE.min(0), "--", alpha=0.4, label="min")
ax[1, 0].plot(a, errREE.max(0), "--", alpha=0.4, label="max")
ax[1, 0].set_title("relative Euler error (signed)")
ax[1, 0].set_xlabel("age group / cohort h")
ax[1, 0].set_ylabel("relative Euler error"); ax[1, 0].legend(fontsize=8)

# Euler equation LHS vs RHS (levels, natural scale): should overlap where slack
ax[1, 1].plot(a, LHS.mean(0), color="k", label="LHS:  1/c")
ax[1, 1].plot(a, LHS.min(0), color="k", ls="--", alpha=0.3)
ax[1, 1].plot(a, LHS.max(0), color="k", ls="--", alpha=0.3)
ax[1, 1].plot(a, RHS.mean(0), color="r", label="RHS:  beta E[(1-d+r')/c']")
ax[1, 1].plot(a, RHS.min(0), color="r", ls="--", alpha=0.3)
ax[1, 1].plot(a, RHS.max(0), color="r", ls="--", alpha=0.3)
ax[1, 1].set_title("Euler equation: LHS vs RHS")
ax[1, 1].set_xlabel("age group / cohort h"); ax[1, 1].legend(fontsize=8)

# saving policy: next-period capital carried by cohort h vs capital held now
for h in range(H - 1):
    ax[1, 2].scatter(k_np[:, h], sav[:, h], s=2, alpha=0.3, label=f"h={h}")
ax[1, 2].set_title("saving policy by cohort")
ax[1, 2].set_xlabel("capital held (current)")
ax[1, 2].set_ylabel("capital saved (next period)"); ax[1, 2].legend(fontsize=8)

ax[1, 3].axis("off")
fig.tight_layout()
plt.show()

The panels match the Exercise-4 reference: a **hump-shaped capital profile**
($k^0=0$ for the newborn, rising through working life, peaking near retirement,
then dissaved), a **smooth rising consumption** profile, a saving policy that
clusters by cohort, a **signed relative Euler error centered on zero** (drawn on
its own small scale, with the zero line — that small magnitude *is* the
accuracy, not noise), and a **Euler LHS that tracks the RHS** wherever the
borrowing constraint is slack. This is the standard life-cycle picture the
course notebook produces — recovered here by the JAX port under the two-stage
loss.

## 5. Ergodic accuracy certificate + the borrowing-constraint corner

We summarize accuracy with **quantiles** of the dimensionless residual
$|\text{err}_{REE}|$ (the Fischer-Burmeister residual is already unit-free) — the
median, 90th, and 99th percentiles over the ergodic sample. Quantiles are honest
about the tail: a low median with a fat p99 would flag a few hard states rather
than hiding behind a mean. The youngest cohort is hardest (it sits closest to
the borrowing constraint, where the FB kink lives); older cohorts are tighter.

In [ ]:
abs_err = np.abs(errREE)  # dimensionless FB residual -> a clean accuracy certificate
print("Ergodic accuracy certificate -- |relative Euler error| (dimensionless FB residual)")
print(f"  median : {np.median(abs_err):.2e}")
print(f"  p90    : {np.quantile(abs_err, 0.90):.2e}")
print(f"  p99    : {np.quantile(abs_err, 0.99):.2e}")
print()
print("per-cohort median |errREE| (h=0..4):", np.round(np.median(abs_err, 0), 4))
print()
print("mean capital by age :", np.round(k_np.mean(0), 3))
print("mean consumption    :", np.round(c.mean(0), 3))
near_bind = (sav[:, : H - 1] < 1e-2).mean(0)
print("frac near borrowing constraint (k' < 1e-2), cohort 0..4:", np.round(near_bind, 3))

**On comparing to the reference.** This is a faithful reproduction of the
Geneva Day 2 Ex 4 *method and diagnostics*: same equilibrium conditions, same FB
complementarity, same decade calibration, same diagnostic panels. On a like-for-
like ergodic $|\text{err}_{REE}|$ the JAX port currently trails the original
TensorFlow reference by roughly a part in ten on the log scale (a known gap
tracked in the project notes, not a ship blocker) — the shapes and the
economics agree; the last fraction of a decimal of accuracy is the open item.

The model exposes nothing exotic to get here: standard MLP, sigmoid-bounded
saving rates for $0 < s < 1$ (hence $c>0$ and $k'\ge 0$ by construction), and the
two-stage `inside_fn`/`combine_fn` hooks that put the Fischer-Burmeister
**after** the expectation.

## Summary

- **Model**: 6-generation life-cycle OLG with borrowing constraints (Geneva Day 2
  Ex 4). 7 states, 5 saving-rate policies, 5 Fischer-Burmeister Euler conditions,
  no closed-form steady state.
- **Capability it unlocked**: the **two-stage expectation-inside-residual loss**.
  Because the FB wraps an expectation, $\mathbb{E}[f^{FB}] \ne f^{FB}(\mathbb{E})$;
  `inside_fn` averages the continuation terms and `combine_fn` applies the FB
  afterward. The standard $(\mathbb{E}[r])^2$ path is recovered as
  `combine = identity`.
- **Result**: textbook life-cycle behaviour (capital hump, consumption smoothing)
  with ergodic per-cohort Euler errors in the $10^{-2}$–$10^{-3}$ range,
  reproducing the course exercise's diagnostic panels.
- **No special tooling**: plain MLP + sigmoid bounds + the two model hooks; the
  trainer, loss, and `deqn_jax.plots` suite are all stock.

The companion model with a closed-form oracle is `examples/olg_analytic_6.ipynb`
(Krueger-Kubler 2004); the constrained-labor sibling is
`bm_labor_constrained` (Day 2 Ex 3).